# EDSS 필수 3개 분야 전체 수집 검증

## tl;dr

2026년 8월 제공목록과 EDSS 실시간 다운로드 화면을 기준으로 고등교육통계·대학정보공시·취업통계의 233개 논리 테이블, 265개 물리 다운로드 단위를 모두 확보했다. 기존 및 신규 원본 278개 ZIP의 체크섬과 압축 무결성 검사가 모두 통과했다.

## Context & Methods

공식 Excel 목록의 논리 테이블과 포털 화면의 `domnCd` 다운로드 단위를 구분한다. 같은 표가 연도·스키마에 따라 여러 `domnCd`로 나뉘면 모든 물리 단위를 확보해야 논리 테이블을 완료한 것으로 본다.

### Key Assumptions

- 수집 범위는 고등교육통계, 대학정보공시, 취업통계다.
- 원본 ZIP은 수정하지 않으며 Git에는 포함하지 않는다.
- 완료 조건은 모든 다운로드 단위의 로컬 파일 존재, 기록된 SHA-256 일치, ZIP 무결성 통과다.

## Data

검증 입력은 실시간 다운로드 목록, 기존 원본 매니페스트, 전체 수집 이력, 최종 상태표와 요약 파일이다.

In [1]:
import csv
import json
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
metadata = repo_root / 'data/metadata'
summary = json.loads((metadata / 'edss_full_collection_summary.json').read_text(encoding='utf-8'))
with (metadata / 'edss_full_collection_status.csv').open(encoding='utf-8-sig', newline='') as handle:
    status_rows = list(csv.DictReader(handle))
with (metadata / 'edss_file_manifest.jsonl').open(encoding='utf-8-sig') as handle:
    existing_archives = [json.loads(line) for line in handle if line.strip()]
with (metadata / 'edss_full_collection_attempts.jsonl').open(encoding='utf-8-sig') as handle:
    collection_attempts = [json.loads(line) for line in handle if line.strip()]

summary

{'generated_at': '2026-09-01T03:03:30.301329+00:00',
 'grain': 'one row per live EDSS domn_code in the three required domains',
 'target_count': 265,
 'logical_table_count': 233,
 'downloaded_logical_table_count': 233,
 'downloaded': 265,
 'failed': 0,
 'invalid': 0,
 'pending': 0,
 'by_source': {'고등교육통계': {'target': 133,
   'downloaded': 133,
   'failed': 0,
   'invalid': 0,
   'pending': 0},
  '대학정보공시': {'target': 130,
   'downloaded': 130,
   'failed': 0,
   'invalid': 0,
   'pending': 0},
  '취업통계': {'target': 2,
   'downloaded': 2,
   'failed': 0,
   'invalid': 0,
   'pending': 0}},
 'verified_archive_count': 278,
 'invalid_archive_count': 0,
 'complete': True}

## Results

In [2]:
assert summary['complete'] is True
assert summary['logical_table_count'] == 233
assert summary['downloaded_logical_table_count'] == 233
assert summary['target_count'] == 265
assert summary['downloaded'] == 265
assert summary['failed'] == 0
assert summary['invalid'] == 0
assert summary['pending'] == 0
assert summary['verified_archive_count'] == 278
assert len(status_rows) == 265
assert {row['status'] for row in status_rows} == {'downloaded'}

[{'분야': source, **counts} for source, counts in summary['by_source'].items()]

[{'분야': '고등교육통계',
  'target': 133,
  'downloaded': 133,
  'failed': 0,
  'invalid': 0,
  'pending': 0},
 {'분야': '대학정보공시',
  'target': 130,
  'downloaded': 130,
  'failed': 0,
  'invalid': 0,
  'pending': 0},
 {'분야': '취업통계',
  'target': 2,
  'downloaded': 2,
  'failed': 0,
  'invalid': 0,
  'pending': 0}]

In [3]:
successful_new = [row for row in collection_attempts if row.get('status') == 'downloaded']
failed_new = [row for row in collection_attempts if row.get('status') != 'downloaded']
total_bytes = sum(int(row.get('size_bytes', 0)) for row in existing_archives + successful_new)
{
    '기존 ZIP': len(existing_archives),
    '신규 ZIP': len(successful_new),
    '실패 기록': len(failed_new),
    '전체 ZIP': len(existing_archives) + len(successful_new),
    '전체 크기 GiB': round(total_bytes / 1024**3, 3),
}

{'기존 ZIP': 29, '신규 ZIP': 249, '실패 기록': 0, '전체 ZIP': 278, '전체 크기 GiB': 1.224}

## Takeaways

- 필수 세 분야의 공식 논리 테이블 233개가 모두 수집됐다.
- 포털의 연도·스키마 분할을 포함한 물리 다운로드 단위 265개가 모두 완료됐다.
- 기존 29개와 신규 249개를 합친 278개 ZIP에서 체크섬 불일치나 압축 손상이 발견되지 않았다.
- 이후 개방ID 검사는 기존 15개 선별 데이터가 아니라 이 전체 원본을 대상으로 다시 구축해야 한다.